In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from planet import Session, data_filter, DataClient
from datetime import datetime
import json
import shapely
from shapely.geometry import shape
import pyproj
import ee
# ee.Authenticate()
# ee.Initialize()
import ipyleaflet
import geemap
import shapely.geometry as sg
import numpy as np
import geojson

# INPUT

In [2]:
#area of interest shapefile (should contain only one geometry):

gdf=gpd.read_file('/Users/isabellopez/Desktop/SURGE/notebooks/exploration/data/DISTRICT2011.shp')
aoi_path= gdf[gdf['NAME']=='Jodhpur']

#skysat years (we will only include imagery from March to June. only the most recent images will be used.)
years=[2021, 2022, 2023]

# coverage requirement for final tiles(percent)
crop_coverage_thresh=30

#number of output tiles
tile_n=100

# earth engine crop_class (crops=4)
crop_class=4

overlap_thresh =50


In [3]:
aoi_path

,DIST_ID,NAME,STATE_UT,C_CODE11,TOT_POP,M_POP,F_POP,TOT_L6,M_L6,F_L6,...,P_LIT_POP,P_R_LIT,P_U_LIT,T_P_M_LIT,P_R_M_LIT,P_U_M_LIT,T_P_F_LIT,P_R_F_LIT,P_U_F_LIT,geometry
157,113,Jodhpur,Rajasthan,0811300000000000,3685681,1924326,1761355,592959,313704,279255,...,67.09,59.79,80.23,80.46,76.32,87.81,52.57,41.99,71.85,"POLYGON ((72.26823 27.55385, 72.27026 27.55324..."


---
# CODE
no need to change any code below

## 0. Define utilities

In [4]:
# Utility functions: projecting a feature to the appropriate UTM zone

def get_utm_projection_fcn(geom):
    # define projection
    # from shapely [docs](http://toblerity.org/shapely/manual.html#shapely.ops.transform)
    proj_fcn = partial(
        pyproj.transform,
        pyproj.Proj(init='EPSG:4326'), #wgs84
        pyproj.Proj(_get_utm_projection(geom)))
    return proj_fcn

def _get_utm_zone(geom):
    '''geom: geojson geometry'''
    coordinates = shape(geom)
    lon = coordinates.centroid.x
    lat = coordinates.centroid.y
    
    if lat > 84 or lat < -80:
        raise Exception('UTM Zones only valid within [-80, 84] latitude')
    
    # this is adapted from
    # https://www.e-education.psu.edu/natureofgeoinfo/book/export/html/1696
    zone = int((lon + 180) / 6 + 1)
    
    hemisphere = 'north' if lat > 0 else 'south'
    
    return (zone, hemisphere)

def _get_utm_projection(geom):
    zone,hemisphere= _get_utm_zone(geom)
    if hemisphere =='north':
        proj_str = f"EPSG:326{zone}"
    else:
        proj_str= f"EPSG:327{zone}"
    return proj_str



## 1. Find SkySat coverage

In [5]:
# gdf_shp=gpd.read_file(gdf).geometry.values[0]
aoi_shp=gdf.geometry.values[0]
# aoi_gj= json.loads(shapely.to_geojson(aoi_shp))
def shapely_to_geojson(geometry):
    return geojson.dumps(geometry)
aoi_gj = json.loads(shapely_to_geojson(aoi_shp))
# aoi_gj= json.loads(shapely.to_geojson(aoi_shp))
aoi_crs=_get_utm_projection(aoi_shp)

# create an API Request from the search specifications

item_type = ['SkySatCollect']

geom_filter = data_filter.geometry_filter(aoi_gj)
cloud_cover_filter = data_filter.range_filter('cloud_cover', None, 0.5)
scenes=gpd.GeoDataFrame(pd.DataFrame({'id':[]}), geometry=[],crs='EPSG:4326')
for year in years:
    date_range_filter = data_filter.date_range_filter("acquired", datetime(month=3, day=1, year=year),
                                                      datetime(month=6, day=30, year=year))
    combined_filter = data_filter.and_filter([geom_filter, date_range_filter, cloud_cover_filter])
    async with Session() as sess:
        cl = DataClient(sess)
        items = cl.search(name='calc_coverage_search',search_filter=combined_filter, item_types=item_type, limit=500)
        item_list = [i async for i in items]
        id_list=[i['id'] for i in item_list]
        geom_list=[shape(i['geometry']) for i in item_list]
    
    scenes=pd.concat([scenes, gpd.GeoDataFrame(pd.DataFrame({'id':id_list}), geometry=geom_list,crs='EPSG:4326')])

In [6]:
aoi_crs

'EPSG:32643'

In [7]:
#we do not want a MultiPolygon
# scenes_shp=scenes.geometry.unary_union
scenes

,id,geometry
0,20210629_083200_ssc11_u0002,"POLYGON ((77.47690 35.46244, 77.47693 35.46244..."
1,20210629_052927_ssc2_u0001,"POLYGON ((73.83893 34.23934, 73.83906 34.23396..."
2,20210628_092525_ssc19_u0002,"POLYGON ((73.86181 33.42703, 73.84943 33.41647..."
3,20210628_092525_ssc19_u0003,"POLYGON ((73.94605 33.35634, 73.93385 33.34567..."
4,20210628_092525_ssc19_u0001,"POLYGON ((73.76180 33.51076, 73.74954 33.50006..."
...,...,...
380,20230305_044349_ssc2_u0001,"POLYGON ((75.56338 35.27330, 75.56319 35.27397..."
381,20230305_031221_ssc14_u0001,"POLYGON ((73.43765 34.34714, 73.43758 34.34041..."
382,20230304_090825_ssc6_u0001,"POLYGON ((73.89663 33.35307, 73.89674 33.35304..."
383,20230303_053546_ssc1_u0001,"POLYGON ((73.84491 34.19028, 73.84373 34.18210..."


## 2. Filter scenes to contain only more than 30 percent crops

In [8]:
def scene_filter(scene, crop_class):
     # Define the visualization parameters for the crop layer
    cropVis = {"min":0, "max":1, 'palette': ['black', '#E49635']}

    legend_dict = {'Crops': '#E49635'}

    # resolution
    scale=10

    cropVis2 = {"min":0, "max":1, 'palette': ['black', '#34EBE1']}

    legend_dict2 = {'Crops': '#34EBE1'}
    
    aoi_json=json.loads(shapely_to_geojson(scene['geometry'].values[0]))
    roi = ee.Geometry.Polygon(aoi_json['coordinates'])
    
    landcover = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate("2015-01-1","2023-05-19").filterBounds(roi).select('label')

    # Create a binary layer where 1 represents crop and 0 represents all other landcover types
    landcover_crop = landcover.map(lambda image: image.eq(crop_class))

    crops = landcover_crop.mosaic().clip(roi).selfMask()
    vectors = crops.reduceToVectors(eightConnected=False, geometryType='polygon', 
                                    crs=crops.projection(), scale=scale).union(1).getInfo()
    
    crops_gdf=gpd.GeoDataFrame({'geometry':[shape(vectors['features'][0]['geometry'])]})
    percent_coverage=100*crops_gdf.set_crs(4326).to_crs(32643).area.values[0]/scene.to_crs(aoi_crs).area.values[0]
    
    return gpd.GeoDataFrame({'id':[scene.id.values[0]], 'coverage':[percent_coverage], 'geometry':[scene.geometry.values[0]]})

In [164]:
jaipur_centroid = aoi_path.geometry.centroid
jaipur_centroid

145    POINT (75.71648 26.96829)
dtype: geometry

In [ ]:
Map = geemap.Map()
Map

Map(center=[20, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(Togg…

In [18]:
# a= scene_filter(Map, scenes[:1], crop_class)
# a

In [19]:
# crop_cov=gpd.read_file('data/crop_cov_temp.shp')
crop_cov=gpd.GeoDataFrame({'id':[], 'coverage':[], 'geometry':[]}).set_crs(4326)
for i in range(len(crop_cov), len(scenes)):
    print(f'{i}/{len(scenes)-1}')
    crop_cov=pd.concat([crop_cov,scene_filter(scenes[i:i+1], crop_class)])
    # crop_cov.to_file('crop_cov_temp.shp')
    

0/1273
1/1273
2/1273
3/1273
4/1273
5/1273
6/1273
7/1273
8/1273
9/1273
10/1273
11/1273
12/1273
13/1273
14/1273
15/1273
16/1273
17/1273
18/1273
19/1273
20/1273
21/1273
22/1273
23/1273
24/1273
25/1273
26/1273
27/1273
28/1273
29/1273
30/1273
31/1273
32/1273
33/1273
34/1273
35/1273
36/1273
37/1273
38/1273
39/1273
40/1273
41/1273


EEException: Invalid geometry.

In [357]:
crop_cov

,id,coverage,geometry
0,20210622_051753_ssc3_u0001,4.094752,"POLYGON ((75.79765 26.84055, 75.79775 26.83384..."
1,20210621_082626_ssc10_u0001,3.863074,"POLYGON ((75.78060 26.86223, 75.78074 26.85491..."
2,20210616_054209_ssc17_u0001,4.137158,"POLYGON ((75.79840 26.84188, 75.79843 26.83578..."
3,20210610_015139_ssc15_u0001,11.325827,"POLYGON ((75.82686 26.75873, 75.82692 26.75873..."
4,20210521_004234_ssc19_u0001,3.791677,"POLYGON ((75.78080 26.86385, 75.78088 26.86385..."
5,20210505_082107_ssc8_u0001,11.069189,"POLYGON ((75.83265 26.75305, 75.83269 26.74751..."
6,20210421_060737_ssc14_u0002,12.618217,"POLYGON ((75.73897 26.93681, 75.73909 26.93696..."
7,20210421_060737_ssc14_u0001,5.698053,"POLYGON ((75.85903 26.86658, 75.85921 26.86682..."
8,20210420_063217_ssc15_u0002,11.297733,"POLYGON ((75.74890 26.93159, 75.75460 26.92825..."
9,20210420_063217_ssc15_u0001,6.026572,"POLYGON ((75.86345 26.86430, 75.86927 26.86085..."


In [300]:
chosen_scenes=crop_cov[crop_cov.coverage>=crop_coverage_thresh]

#save chosen scenes into a shapefile
chosen_scenes.to_file(f'chosen_scenes_{crop_coverage_thresh}_percent.shp')

In [301]:
chosen_scenes

,id,coverage,geometry
0,20220524_051051_ssc12_u0004,41.659474,"POLYGON ((75.39937 27.17459, 75.39938 27.17460..."
0,20220524_051051_ssc12_u0003,46.407888,"POLYGON ((75.53464 27.10782, 75.53464 27.10783..."
0,20220524_051051_ssc12_u0002,71.612507,"POLYGON ((75.66967 27.03944, 75.66973 27.03954..."
0,20220524_010936_ssc19_u0001,71.396601,"POLYGON ((76.22125 27.02244, 76.20953 27.01178..."
0,20220524_010936_ssc19_u0002,72.159822,"POLYGON ((76.31487 26.93988, 76.30328 26.92894..."
0,20220519_083856_ssc8_u0001,77.696718,"POLYGON ((76.23630 27.80560, 76.24247 27.80129..."
0,20220519_083856_ssc8_u0002,70.705147,"POLYGON ((76.35902 27.71832, 76.36517 27.71400..."
0,20220428_051006_ssc2_u0002,77.598364,"POLYGON ((76.32596 27.68782, 76.33134 27.68260..."
0,20220428_051006_ssc2_u0001,75.276676,"POLYGON ((76.21941 27.79404, 76.22486 27.78877..."
0,20220411_084414_ssc8_u0006,69.751132,"POLYGON ((76.14560 27.73554, 76.14564 27.73554..."


# 3. Remove Overlapping Scenes

In [333]:
def remove_overlapping_scenes(scenes):
    non_overlapping_scenes = gpd.GeoDataFrame({'id':[], 'coverage':[], 'geometry':[]}).set_crs(4326)
    while len(scenes) > 0:
        scene1 = scenes.iloc[0]
        scenes.drop(0, inplace= True)
        scenes.reset_index(inplace=True)
        scenes.drop(columns='index',inplace= True)
        # overlapping = [i for i in range(len(scenes)) if scene1['geometry'].intersects(scenes['geometry'].values[i])]
        overlapping = [i for i in range(len(scenes)) if 100*(scene1['geometry'].intersection(scenes['geometry'].values[i]).area)/scene1['geometry'].area>overlap_thresh]
        keep=True
        for i in overlapping:
            if scene1['coverage'] > scenes['coverage'].values[i]:  # scene1 has more coverage, so remove scene2
                scenes.drop(i, inplace= True)
            else:  # scene2 has more coverage, so remove scene1 and break the loop
                keep= False
                break
        if keep:
            non_overlapping_scenes=pd.concat([non_overlapping_scenes,
                                              gpd.GeoDataFrame({'id':[scene1['id']], 
                                                                'coverage':[scene1['coverage']],
                                                                'geometry':[scene1['geometry']]}).set_crs(4326)])
        scenes.reset_index(inplace=True)
        scenes.drop(columns='index',inplace= True)
    return non_overlapping_scenes


In [335]:
chosen_scenes = gpd.read_file(f'data/chosen_scenes_{crop_coverage_thresh}_percent.shp')
chosen_scenes

,id,coverage,geometry
0,20220524_051051_ssc12_u0004,41.659474,"POLYGON ((75.39937 27.17459, 75.39938 27.17460..."
1,20220524_051051_ssc12_u0003,46.407888,"POLYGON ((75.53464 27.10782, 75.53464 27.10783..."
2,20220524_051051_ssc12_u0002,71.612507,"POLYGON ((75.66967 27.03944, 75.66973 27.03954..."
3,20220524_010936_ssc19_u0001,71.396601,"POLYGON ((76.22125 27.02244, 76.20953 27.01178..."
4,20220524_010936_ssc19_u0002,72.159822,"POLYGON ((76.31487 26.93988, 76.30328 26.92894..."
5,20220519_083856_ssc8_u0001,77.696718,"POLYGON ((76.23630 27.80560, 76.24247 27.80129..."
6,20220519_083856_ssc8_u0002,70.705147,"POLYGON ((76.35902 27.71832, 76.36517 27.71400..."
7,20220428_051006_ssc2_u0002,77.598364,"POLYGON ((76.32596 27.68782, 76.33134 27.68260..."
8,20220428_051006_ssc2_u0001,75.276676,"POLYGON ((76.21941 27.79404, 76.22486 27.78877..."
9,20220411_084414_ssc8_u0006,69.751132,"POLYGON ((76.14560 27.73554, 76.14564 27.73554..."


In [337]:
chosen_scenes = remove_overlapping_scenes(chosen_scenes)

#save chosen scenes that are not overlapping into a shapefile
# chosen_scenes.to_file(f'no_overlapping_{crop_coverage_thresh}_percent.shp')

In [338]:
chosen_scenes.explore()

In [331]:
scene1=scenes.geometry.values[0]
scene2=scenes.geometry.values[1]
100*(scene1.intersection(scene2)).area/scene1.area

84.58992557971649

In [329]:
scene1.area

0.0075501605825293435

In [330]:
scene2.area

0.00741651992704897